# MLP Model Multivariate

In this section we implement multivariate forecasting using the MLP Model with the **TimeSeriesDatasetVectorizedExog** approach.

The MLP (Multi-Layer Perceptron) Forecaster is the same univariate model used in the univariate approach, but extended to multivariate forecasting through efficient batching. Instead of processing series individually, **fTimeSeriesDatasetVectorizedExog** batches all 1502 series together, allowing the univariate model to train on multiple series simultaneously with exogenous features (GDP, CPI, Interest Rate).

The model architecture remains unchanged - we simply reshape the data to process all series in parallel, achieving 585x faster training while incorporating exogenous variables.


**Layer Breakdown:**

- **Input Flattening**: Reshapes (batch_size, seq_length, input_size) → (batch_size, seq_length × input_size)
- **Hidden Layers**: 3 stacked fully connected layers with ReLU activation
- **Hidden Size**: 512 units per layer
- **Dropout**: Applied after each hidden layer (0.2)
- **Output Layer**: Single fully connected layer producing 1-step forecast

In [1]:
import torch
import torch.nn as nn

## Model

In [ ]:
class MLPForecaster(nn.Module):
    """
    MLP model for UNIVARIATE time series forecasting with one-hot encoding.
    Architecture: 
        Input Flattening -> MLP Layers (Linear -> ReLU -> Dropout) -> Output
    
    Takes input sequences with multiple features (Value + features + year + month + one-hot)
    and flattens them before processing through fully connected layers.
    
    Unlike MLPMultivariate:
    - Uses one-hot encoding to identify individual time series
    - Predicts one value at a time for a specific series
    - Processes entire sequence as flattened vector
    
    Good for:
    - Capturing non-linear patterns across the entire sequence
    - Faster training than RNN/LSTM for shorter sequences
    - Learning complex feature interactions
    """
    def __init__(self, input_size, seq_length, hidden_size=512, num_layers=3, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            seq_length: Sequence length (lookback window)
            hidden_size: Number of units in each MLP layer
            num_layers: Number of MLP layers
            dropout: Dropout rate
        """
        super(MLPForecaster, self).__init__()
        
        self.input_size = input_size
        self.seq_length = seq_length
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Calculate input dimension after flattening
        self.input_dim = seq_length * input_size
        
        # Build MLP layers
        layers = []
        
        # First layer
        layers.append(nn.Linear(self.input_dim, hidden_size))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        
        # Hidden layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
        
        self.mlp = nn.Sequential(*layers)
        
        # Output layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        """
        Args:
            x: Input tensor of shape (batch_size, seq_length, input_size)
        
        Returns:
            predictions: (batch_size, 1) - single value prediction
        """
        batch_size = x.size(0)
        
        # Flatten entire sequence
        x_flat = x.reshape(batch_size, -1)  # (batch_size, seq_length * input_size)
        
        # Pass through MLP
        x = self.mlp(x_flat)  # (batch_size, hidden_size)
        
        # Output prediction
        out = self.fc(x)  # (batch_size, 1)
        
        return out


### Model Results without Exogenous Features

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration |
|-------|----------------|------------|---------|-------------|---------------|------------|----------|
| 0 | 0.1426 | 16 | 0.226 | 128 | 0.0001 | 2 | 0.85s |
| 1 | 0.1235 | 8 | 0.331 | 512 | 0.0068 | 4 | 1.97s |
| 2 | 0.1260 | 8 | 0.137 | 128 | 0.0003 | 3 | 0.82s |
| 3 | 0.1320 | 8 | 0.393 | 256 | 0.0007 | 2 | 0.31s |
| 4 | 0.1226 | 4 | 0.135 | 128 | 0.0029 | 2 | 0.68s |


#### Best Hyperpareters

Validation Loss: 0.122618

Parameters:
  - learning_rate: 0.00286
  - batch_size: 4
  - num_layers: 2
  - hidden_size: 128
  - dropout: 0.134

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/mlp/fold1/fold_results.png)


#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/mlp/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

    
![Fold 3 Results](./img/multivariate/mlp/fold3/fold_results.png)

### Fold Results

| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|------|-------|
| Fold 1 | 56527.62 | 237.76 | 106.15 | 0.8880 | 87.83% |
| Fold 2 | 50669.06 | 225.10 | 100.61 | 0.8881 | 83.89% |
| Fold 3 | 48325.29 | 219.83 | 93.74 | 0.9006 | 86.13% |
| **Average** | **51840.65 ± 4224.81** | **227.56 ± 9.21** | **100.17 ± 6.22** | **0.8922 ± 0.0073** | **85.95% ± 1.97%** |


#### Average SMAPE Distribution Across Folds

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 2.9% ± 0.2% | 44 |
| 10-20% | 11.3% ± 1.0% | 169 |
| 20-30% | 13.3% ± 2.1% | 200 |
| 30-40% | 10.0% ± 0.8% | 150 |
| >40% | 62.5% ± 3.9% | 939 |


**Comparison with Baseline:**

The MLP multivariate model achieves an average SMAPE of 85.95% ± 1.97%, which is **13.69 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%). This indicates that the baseline model performs better on average. However, the MLP model shows more consistent predictions across folds (lower standard deviation: 1.97% vs 7.06%), suggesting more stable performance despite higher overall error.

### Model Results with Exogenous Features


### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration |
|-------|----------------|------------|---------|-------------|---------------|------------|----------|
| 0 | 0.3024 | 4 | 0.499 | 512 | 0.0024 | 3 | 2.68s |
| 1 | 0.1941 | 16 | 0.375 | 1024 | 0.0013 | 2 | 3.86s |
| 2 | 0.1931 | 16 | 0.193 | 256 | 0.0078 | 3 | 1.15s |
| 3 | 0.2956 | 4 | 0.118 | 256 | 0.0013 | 3 | 0.47s |
| 4 | 0.2203 | 4 | 0.253 | 128 | 0.0031 | 3 | 0.91s |

### Best Hyperparameters 

Parameters:
  - learning_rate: 0.007837558806870324
  - batch_size: 16
  - num_layers: 3
  - hidden_size: 256
  - dropout: 0.19275252335431695


#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/mlp_exog/fold1/fold_results.png)


#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/mlp_exog/fold2/fold_results.png)


#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30
    
![Fold 3 Results](./img/multivariate/mlp_exog/fold3/fold_results.png)

#### Fold Results 

| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|------|-------|
| Fold 1 | 119352.35 | 345.47 | 180.57 | 0.7635 | 117.66% |
| Fold 2 | 80240.07 | 283.27 | 204.55 | 0.8228 | 120.10% |
| Fold 3 | 137209.79 | 370.42 | 334.73 | 0.7179 | 131.35% |
| **Average** | **112267.40 ± 29219.28** | **333.04 ± 44.72** | **239.95 ± 82.63** | **0.7681 ± 0.0526** | **123.04% ± 7.10%** |

#### Average SMAPE Distribution Across Folds

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 1.3% ± 0.6% | 19 |
| 10-20% | 5.3% ± 2.1% | 80 |
| 20-30% | 6.1% ± 1.3% | 92 |
| 30-40% | 4.8% ± 0.5% | 73 |
| >40% | 82.4% ± 4.2% | 1238 |

**Comparison with Baseline:**

The MLP multivariate model with exogenous features achieves an average SMAPE of 123.04% ± 7.10%, which is **50.78 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%). This indicates that the baseline model significantly outperforms the MLP model with exogenous features. The addition of exogenous variables (GDP, CPI, Interest Rate) appears to have degraded model performance rather than improving it, with the exogenous model performing 37.09 percentage points worse than the MLP model without exogenous features (85.95% ± 1.97%). Both models show similar consistency across folds (standard deviation: 7.10% vs 7.06%), but the substantially higher error suggests that the exogenous features may not be effectively captured by the current model architecture.